# No-show Prediction

This notebook analyzes healthcare appointment data to predict patient no-shows. The workflow includes:

- Reading and loading the dataset
- Data cleaning and schema definition
- Handling missing values and dropping unnecessary columns
- Identifying and scaling numeric features for modeling

The goal is to prepare the data for building predictive models to identify factors influencing patient attendance.


9-7: 10:20am modification: Move FE pipeine to separate notebook, writing to Delta object.

### Setup cells commented out for use in Jobs pipeline

In [0]:
%skip
%run ./setup/import_dataset
%run ./setup/data_preprocessing

In [0]:
### Read features from a table

train_scaled = spark.table("default.no_show_train_scaled_tbl")
test_scaled = spark.table("default.no_show_test_scaled_tbl")

In [0]:
print(f"Features read in from Delta Table")
print(f"  Train: {train_scaled.count():,} rows, {len(train_scaled.columns)} columns")
print(f"  Test: {test_scaled.count():,} rows, {len(test_scaled.columns)} columns")

#### Hyperparameter tuning

#### Apply hyperparameter grid
ParamGridBuilder: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.ParamGridBuilder.html

In [0]:
import numpy as np
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel

In [0]:
# Scorer
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)


### CrossValidator Documentation
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/no_show_volume"

In [0]:
%skip
from sklearn.model_selection import GridSearchCV
clf = GridSearchCV(lr, param_grid = rf_model, lr_cv=3, verbose = True, n_jobs = 1)
clf

In [0]:
%skip
# best_clf = clf.fit(x,y) -- featuresCol, labelCol
best_clf = clf.fit(features, Showed_up)
best_clf = clf.best_estimator_

### Random Forest Classifier

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol = 'scaledFeatures',
                            labelCol = 'Showed_up',
                            predictionCol = 'prediction',
                            weightCol = 'weightCol'
)

#### Create RF Param grid

In [0]:
%skip
rf_param_grid = (ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 200]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .addGrid(rf.minInstancesPerNode, [1, 10]) \
    .build()
) # End param_grid

In [0]:
%skip
rf_param_grid = (ParamGridBuilder() \
    .addGrid(rf.numTrees, [75, 100]) \
    .addGrid(rf.maxDepth, [5, 15]) \
    .build()
) # End param_grid

In [0]:
rf_param_grid = (ParamGridBuilder() \
    .addGrid(rf.numTrees, [25]) \
    .addGrid(rf.maxDepth, [5]) \
    .build()
) # End param_grid

### HTuning notes

Changing maxDepth from (5,10) to (5,15) increased f1 by 13%. Chnging numtrees to 75,150 was negligible.

#### RF CrossValidator

In [0]:
cv_rf = CrossValidator(
    estimator = rf,
    estimatorParamMaps = rf_param_grid,
    evaluator = evaluator,
    numFolds = 3,
    parallelism = 2
)

In [0]:
%skip
# pipeline = Pipeline(stages=[indexer, encoder, vector_assembler])

stages2=[indexer, encoder, vector_assembler, rf]
pipeline2 = Pipeline().setStages(stages2)

In [0]:
train_df = spark.read.table('train_df_tbl')
test_df = spark.read.table('test_df_tbl')

In [0]:
%skip
# train_scaled already has indexing/encoding applied, fit LR directly
from pyspark.ml.classification import LogisticRegression

lr_model = lr.fit(train_scaled)
lr_predictions = lr_model.transform(test_scaled)

In [0]:
%skip
import gc

# Free model cache from previous LR and RF CrossValidator runs
for _v in ("cvModel", "lr_model", "lr_predictions", "rf_model", "rf_predictions"):
    if _v in globals():
        del globals()[_v]
gc.collect()

In [0]:
# train_scaled already has indexing/encoding applied, fit RF directly
rf_model = cv_rf.fit(train_scaled)
rf_predictions = rf_model.transform(test_scaled)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

In [0]:
import mlflow       # Experiment tracking & Record ML runs
import mlflow.spark # Log model artifacts
import os
from mlflow.models import infer_signature

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/no_show_volume"

# EXPERIMENT_NAME = "/Users/asanders4205@gmail.com/predict_no_show"
# TARGET = "Showed_up"

In [0]:
#  columns: prediction, label, weight (optional) and probabilityCol (only for logLoss)
multi_evaluator = MulticlassClassificationEvaluator(
    predictionCol = 'prediction',
    labelCol = 'Showed_up'
)

In [0]:
signature = infer_signature(train_df, rf_predictions)

### Evaluate and log RF metrics

In [0]:
# help(RandomForestClassifier)
# print(multi_evaluator.explainParams())
precision = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "weightedPrecision"})
recall = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "weightedRecall"})
f1 = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "f1"})
# auc = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "AreaUnderAUC"})
# roc = multi_evaluator.evaluate(rf_predictions,{multi_evaluator.metricName: "areaUnderROC"})
# accuracy = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "accuracy"})
# support = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "support"})

In [0]:
# Use the signature already inferred in cell 70 (train_df → rf_predictions)
# input_example would fail here due to non-JSON-serializable Vector columns

with mlflow.start_run(run_name="RandomForestModel") as run:
    # Log model artifact with pre-computed signature
    mlflow.spark.log_model(rf_model, 
                           "random_forest_model",
                           signature=signature  # From cell 70
    ) # End log model
    mlflow.log_param("pip_requirements", ["pyspark==4.1.0"])
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("featuresCol", "features")
    mlflow.log_param("labelCol", "Showed_up")
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_param("run_description", "HP tuning: Numtrees 75, 100 - maxDepth 5,15")

### Log Hyperparameters